# Chroma: Local vs Remote Testing

Tests every customer path for the Chroma integration:

| Mode | How | BM25? | Pickle-safe? | Use case |
|------|-----|-------|-------------|----------|
| **In-memory** | `ChromaChunkSource(collection_name=...)` | No | No | Quick prototyping, unit tests |
| **Persistent local** | `ChromaChunkSource(..., path="/tmp/db")` | No | No | Local dev, data persists across restarts |
| **Client-server** | `ChromaChunkSource(..., host="localhost")` | Auto | Yes | Production, training envs |
| **Custom embed_fn** | Any mode + `embed_fn=my_fn` | Depends | Depends | Azure OpenAI, custom models |

Run each section independently to validate the path you care about.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
print(f"Python: {sys.executable}")

import cgft
import chromadb
print(f"cgft: {cgft.__file__}")
print(f"chromadb: {chromadb.__version__}")

In [ ]:
import tempfile, os, shutil
from cgft.corpus.chroma.source import ChromaChunkSource
from cgft.corpus.search_schema.search_types import SearchSpec, FieldPredicate

def create_sample_docs():
    """Create temp dir with sample markdown docs."""
    docs_dir = tempfile.mkdtemp(prefix="chroma_test_")
    docs = {
        "getting_started.md": (
            "# Getting Started\n\n## Installation\n\n"
            "Install with pip:\n\n```bash\npip install cgft\n```\n\n"
            "Python 3.12 required. Python 3.13 is not supported due to "
            "a pathlib.Path pickle incompatibility.\n\n"
            "## Quick Start\n\nCreate a project directory and initialize "
            "your first experiment. The system will automatically chunk "
            "your documents, compute embeddings, and upload them to "
            "ChromaDB for retrieval during training.\n"
        ),
        "features/search.md": (
            "# Search Features\n\n## Vector Search\n\n"
            "Dense vector similarity uses cosine distance by default. "
            "You can provide a custom embedding function or use Chroma's "
            "built-in sentence-transformers model.\n\n"
            "### Supported Distance Metrics\n\n"
            "- cosine (default) - angular similarity\n"
            "- l2 - euclidean distance\n"
            "- ip - inner product\n\n"
            "## Hybrid Search\n\n"
            "Combine BM25 and vector search using Reciprocal Rank Fusion.\n"
        ),
        "features/envs.md": (
            "# RL Environments\n\n## Overview\n\n"
            "Environments define how the RL agent interacts with your "
            "corpus during training. Each environment provides tools "
            "the agent can use and a reward function.\n\n"
            "## ChromaSearchEnv\n\n"
            "Extends SearchEnv for ChromaDB. Supports vector, lexical, "
            "and hybrid search modes. Requires client-server mode.\n"
        ),
    }
    for name, content in docs.items():
        fp = os.path.join(docs_dir, name)
        os.makedirs(os.path.dirname(fp), exist_ok=True)
        with open(fp, "w") as f:
            f.write(content)
    return docs_dir


def run_source_tests(source, label):
    """Run standard test suite against any ChromaChunkSource."""
    caps = source.get_search_capabilities()
    print(f"  modes: {caps['modes']}")
    print(f"  client-server: {source.is_client_server}")

    # Total count
    total = source._get_total_count()
    print(f"  total chunks: {total}")
    assert total > 0, "No chunks!"

    # Sample
    sampled = source.sample_chunks(n=2)
    assert len(sampled) == 2, f"Expected 2, got {len(sampled)}"
    print(f"  sample_chunks: OK ({len(sampled)})")

    # Vector search
    results = source.search(SearchSpec(mode="vector", top_k=3, text_query="installation"))
    assert len(results) > 0
    print(f"  vector search: OK ({len(results)} results)")

    # search_content
    content = source.search_content(SearchSpec(mode="vector", top_k=2, text_query="test"))
    assert all(isinstance(s, str) for s in content)
    print(f"  search_content: OK ({len(content)} strings)")

    # search_text
    text_results = source.search_text("distance metrics", top_k=2)
    assert len(text_results) > 0
    print(f"  search_text: OK ({len(text_results)} results)")

    # File awareness
    file_aware = source._files.check()
    print(f"  file-aware: {file_aware}")
    if file_aware:
        paths = source._files.get_all_file_paths()
        print(f"  file paths: {paths}")
        top = source.get_top_level_chunks()
        print(f"  top-level chunks: {len(top)}")
        ctx = source.get_chunk_with_context(sampled[0])
        print(f"  context: prev={'yes' if ctx['prev_chunk_preview'] not in ('', '(No previous chunk)') else 'no'}, next={'yes' if ctx['next_chunk_preview'] not in ('', '(No next chunk)') else 'no'}")

    # search_related
    related = source.search_related(sampled[0], ["install", "setup"], top_k=2)
    print(f"  search_related: {len(related)} results")

    # Filtered search
    filt = source.search(SearchSpec(
        mode="vector", top_k=3, text_query="search",
        filter=FieldPredicate(field="h1", op="eq", value="Search Features"),
    ))
    for c in filt:
        assert c.get_metadata("h1") == "Search Features"
    print(f"  filtered search: OK ({len(filt)} results, all h1='Search Features')")

    # Lexical / hybrid if available
    if "lexical" in caps["modes"]:
        lex = source.search(SearchSpec(mode="lexical", top_k=2, text_query="pip install"))
        print(f"  lexical search: OK ({len(lex)} results)")
        hyb = source.search(SearchSpec(mode="hybrid", top_k=2, text_query="reward function"))
        print(f"  hybrid search: OK ({len(hyb)} results)")
    else:
        print(f"  lexical/hybrid: N/A (vector-only)")

    print(f"\n  [{label}] ALL PASSED")
    return sampled


print("Helpers loaded.")

---
## Mode 1: In-Memory (Ephemeral)

No server, no persistence. Data lives in memory only. Good for quick prototyping.

- No BM25 (vector-only)
- Not pickle-safe (can't be used in training envs)

In [ ]:
print("=" * 50)
print("MODE 1: IN-MEMORY")
print("=" * 50)

docs_dir = create_sample_docs()

source_mem = ChromaChunkSource(
    collection_name="test-in-memory",
    # No host, no path → ephemeral in-memory
)
source_mem.populate_from_folder(
    docs_dir, min_chars=100, max_chars=500, overlap_chars=30, show_summary=False
)

run_source_tests(source_mem, "IN-MEMORY")

# Verify NOT pickle-safe (data would be lost)
import pickle
data = pickle.dumps(source_mem)
restored = pickle.loads(data)
assert restored._client is None, "Client should be stripped"
print(f"\n  Pickle: serialized ({len(data)}B) but data is LOST after restore")
print("  (This is expected — in-memory mode is not for training envs)")

shutil.rmtree(docs_dir)

---
## Mode 2: Persistent Local

Data persists to disk. No server needed. Good for local development.

- No BM25 (vector-only)
- Not pickle-safe for remote envs (path is local)

In [ ]:
print("=" * 50)
print("MODE 2: PERSISTENT LOCAL")
print("=" * 50)

docs_dir = create_sample_docs()
db_dir = tempfile.mkdtemp(prefix="chroma_persist_")

source_persist = ChromaChunkSource(
    collection_name="test-persistent",
    path=db_dir,
)
source_persist.populate_from_folder(
    docs_dir, min_chars=100, max_chars=500, overlap_chars=30, show_summary=False
)

run_source_tests(source_persist, "PERSISTENT LOCAL")

# Verify data persists: create a NEW source pointing to same path
source_persist2 = ChromaChunkSource(
    collection_name="test-persistent",
    path=db_dir,
)
count2 = source_persist2._get_total_count()
print(f"\n  Persistence: reopened same path, found {count2} chunks")
assert count2 == source_persist._get_total_count()
print("  Persistence PASSED")

shutil.rmtree(docs_dir)
shutil.rmtree(db_dir)

---
## Mode 3: Client-Server

Connects to a running Chroma server. **Required for training envs.**

- BM25 available if server supports sparse indexing (auto-detected)
- Pickle-safe (reconnects after deserialize)

**Prerequisites**: Start a Chroma server first:
```bash
chroma run --host localhost --port 8000 --path /tmp/chroma_data
```

In [ ]:
# Set your server details here
CHROMA_HOST = "localhost"
CHROMA_PORT = 8000

In [ ]:
print("=" * 50)
print("MODE 3: CLIENT-SERVER")
print("=" * 50)

# Verify server is reachable
client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
hb = client.heartbeat()
print(f"Server heartbeat: {hb}")

docs_dir = create_sample_docs()

source_server = ChromaChunkSource(
    collection_name="test-client-server",
    host=CHROMA_HOST,
    port=CHROMA_PORT,
    enable_bm25=True,
)
source_server.populate_from_folder(
    docs_dir, min_chars=100, max_chars=500, overlap_chars=30, show_summary=False
)

run_source_tests(source_server, "CLIENT-SERVER")

# Verify pickle roundtrip + reconnect
print("\n--- Pickle Roundtrip ---")
import pickle
data = pickle.dumps(source_server)
restored = pickle.loads(data)
r2 = restored.search(SearchSpec(mode="vector", top_k=2, text_query="installation"))
print(f"  Serialized: {len(data)}B")
print(f"  After restore: {len(r2)} results from reconnected client")
assert len(r2) > 0
print("  Pickle roundtrip PASSED")

# Cleanup
shutil.rmtree(docs_dir)
client.delete_collection("test-client-server")

---
## Mode 4: Custom Embedding Function

Any mode can use a custom `embed_fn`. This test uses a mock (random vectors) to validate the wiring without needing real API credentials.

For Azure OpenAI, replace the mock with:
```python
from openai import AzureOpenAI
_openai = AzureOpenAI(api_key="...", api_version="2024-02-01", azure_endpoint="...")
def embed_fn(texts):
    response = _openai.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]
```

In [ ]:
import numpy as np

print("=" * 50)
print("MODE 4: CUSTOM EMBED_FN (mock)")
print("=" * 50)

EMBED_DIM = 384  # same as all-MiniLM-L6-v2

def mock_embed_fn(texts):
    """Deterministic mock embedding for testing."""
    vecs = []
    for text in texts:
        seed = hash(text) % (2**31)
        rng = np.random.RandomState(seed)
        vec = rng.randn(EMBED_DIM).astype(np.float32)
        vec = vec / np.linalg.norm(vec)  # normalize
        vecs.append(vec.tolist())
    return vecs

# Test with in-memory + custom embed_fn
docs_dir = create_sample_docs()

source_custom = ChromaChunkSource(
    collection_name="test-custom-embed",
    embed_fn=mock_embed_fn,
)
source_custom.populate_from_folder(
    docs_dir, min_chars=100, max_chars=500, overlap_chars=30, show_summary=False
)

run_source_tests(source_custom, "CUSTOM EMBED_FN")

# Verify embed_query uses our function
vec = source_custom.embed_query("test query")
assert vec is not None
assert len(vec) == EMBED_DIM
print(f"\n  embed_query: returned {len(vec)}-dim vector")

# Verify dimension validation catches mismatches
print("  Dimension validation:")
try:
    source_bad = ChromaChunkSource(
        collection_name="test-bad-dim",
        embed_fn=lambda texts: [[1.0, 2.0, 3.0] for _ in texts],  # 3-dim
    )
    source_bad.embed_query("first")  # sets expected dim = 3
    # Now change the function to return different dim
    source_bad._embed_fn = lambda texts: [[1.0, 2.0] for _ in texts]  # 2-dim
    source_bad.embed_query("second")  # should raise
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"    Correctly caught: {e}")
print("  Dimension validation PASSED")

shutil.rmtree(docs_dir)

---
## Mode 5: ChromaSearchEnv (Training Environment)

Tests the RL environment that wraps ChromaChunkSource. Requires client-server mode.

Validates:
- Host is required (raises on empty)
- Tool listing and execution
- Mode selection (vector-only or multi-mode)
- Dataset preprocessing

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from cgft.envs.chroma_search_env import ChromaSearchEnv

print("=" * 50)
print("MODE 5: ChromaSearchEnv")
print("=" * 50)

# --- Host required ---
try:
    ChromaSearchEnv(collection_name="test", host="")
    assert False, "Should have raised"
except ValueError as e:
    print(f"Host validation: {e}")
print("Host validation PASSED\n")

# --- Create env with our server source ---
# Re-use source_server from Mode 3 if available, otherwise use in-memory with mock
try:
    _ = source_server._get_total_count()
    print("Using client-server source from Mode 3")
    env = ChromaSearchEnv(
        collection_name="test-client-server",
        host=CHROMA_HOST,
        port=CHROMA_PORT,
    )
except Exception:
    print("No server available — creating env with mock source")
    from unittest.mock import patch
    with patch("cgft.corpus.chroma.source._has_search_api", return_value=False):
        env = ChromaSearchEnv(
            collection_name="test-env",
            host="localhost",
        )
    env._source = source_mem  # swap in in-memory source

print(f"Default mode: {env._default_mode}")

# --- Tools ---
tools = asyncio.run(env.list_tools())
for t in tools:
    print(f"Tool: {t.name} — {t.description}")
    props = t.input_schema.get("properties", {})
    if "mode" in props:
        print(f"  Modes: {props['mode']['enum']}")
print("Tool listing PASSED\n")

# --- Search execution ---
result = asyncio.run(env._search_tool(query="how to install"))
assert not result.startswith("Error"), f"Search failed: {result[:200]}"
print(f"Search result preview: {result[:200]}...")
print("Search execution PASSED\n")

# --- Empty query ---
empty = asyncio.run(env._search_tool(query=""))
assert "Error" in empty
print("Empty query rejection PASSED\n")

# --- Preprocessing ---
std = ChromaSearchEnv.dataset_preprocess({"question": "Q?", "answer": "A"})
assert std["prompt"] == "Q?" and std["ground_truth"] == "A"
print("Dataset preprocessing PASSED")

print(f"\n=== ChromaSearchEnv ALL PASSED ===")

---
## Summary

In [ ]:
print("=" * 50)
print("CHROMA LOCAL vs REMOTE — SUMMARY")
print("=" * 50)
print()
print("Mode 1 (In-memory):       vector-only, no persistence, not pickle-safe")
print("Mode 2 (Persistent local): vector-only, persists to disk, not remote-safe")
print("Mode 3 (Client-server):    vector + optional BM25, pickle-safe, training-ready")
print("Mode 4 (Custom embed_fn):  works with any mode, dimension validation built-in")
print("Mode 5 (ChromaSearchEnv):  requires client-server, tool execution validated")
print()
print("All modes tested successfully.")